# Fine-tune a binary classification head

SMS spam vs ham. Keep frozen GPT-2, add `Linear(768 → 2)`, train it on the last token hidden state.

## 0. Setup

In [1]:
import csv
import urllib.request
import zipfile
from pathlib import Path

import torch
import torch.nn as nn
from transformers import GPT2LMHeadModel, GPT2Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(123)
print(device)

cpu


## 1. Load GPT-2 and freeze it

`lm_head` is 768 → vocab. Freeze `transformer` so those weights do not train.

In [2]:
tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token

lm = GPT2LMHeadModel.from_pretrained("openai-community/gpt2").to(device)

for p in lm.transformer.parameters():
    p.requires_grad = False
lm.transformer.eval()

print("transformer trainable:", any(p.requires_grad for p in lm.transformer.parameters()))
print("lm_head trainable    :", any(p.requires_grad for p in lm.lm_head.parameters()))
lm.lm_head

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

transformer trainable: False
lm_head trainable    : False


Linear(in_features=768, out_features=50257, bias=False)

In [3]:
lm

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

## 2. Replace the last layer

Swap `lm_head` (768 → vocab) for `Linear(768 → 2)`. New layer is trainable by default.

In [4]:
lm.lm_head = nn.Linear(768, 2).to(device)

print("transformer trainable:", any(p.requires_grad for p in lm.transformer.parameters()))
print("lm_head trainable    :", any(p.requires_grad for p in lm.lm_head.parameters()))
for n, p in lm.named_parameters():
    if p.requires_grad:
        print("trainable:", n, tuple(p.shape))
lm.lm_head

transformer trainable: False
lm_head trainable    : True
trainable: lm_head.weight (2, 768)
trainable: lm_head.bias (2,)


Linear(in_features=768, out_features=2, bias=True)

## 3. Data

UCI SMS: `ham=0`, `spam=1`.

In [5]:
data_dir = Path("notebooks/LLM_Scratch/data")
if not data_dir.parent.exists():
    data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
tsv_path = data_dir / "SMSSpamCollection"

if not tsv_path.exists():
    zip_path = data_dir / "sms.zip"
    urllib.request.urlretrieve(
        "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip",
        zip_path,
    )
    zipfile.ZipFile(zip_path).extractall(data_dir)

texts, labels = [], []
with tsv_path.open(encoding="utf-8") as f:
    for lab, text in csv.reader(f, delimiter="\t"):
        texts.append(text)
        labels.append(0 if lab == "ham" else 1)
        if len(labels) > 50:
            break
print(len(texts), "ham", labels.count(0), "spam", labels.count(1))

51 ham 41 spam 10


In [6]:
texts[0], labels[0]

('Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...',
 0)

## 4. Tokenize and split

In [7]:
MAX_LEN = 32
enc = tokenizer(texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
token_ids, mask = enc["input_ids"], enc["attention_mask"]
y = torch.tensor(labels)

split = torch.randperm(len(y))
n_val = int(0.2 * len(y))
val_idx, train_idx = split[:n_val], split[n_val:]

token_ids_tr, mask_tr, y_tr = token_ids[train_idx], mask[train_idx], y[train_idx]
token_ids_va, mask_va, y_va = token_ids[val_idx], mask[val_idx], y[val_idx]

from torch.utils.data import DataLoader, TensorDataset

BATCH_SIZE = 8
train_loader = DataLoader(TensorDataset(token_ids_tr, mask_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(token_ids_va, mask_va, y_va), batch_size=BATCH_SIZE)

batch_ids, batch_mask, batch_y = next(iter(train_loader))
print(token_ids_tr.shape, "-> batch", batch_ids.shape, batch_y.shape)

torch.Size([41, 32]) -> batch torch.Size([8, 32]) torch.Size([8])


In [8]:
token_ids[0], mask[0]

(tensor([ 5247,  1566,  8174,   506,   966,    11,  7165,   492, 14898,   691,
           287,  5434,   271,   299,  1049,   995,  8591,   304, 44703,   986,
           327,   500,   612,  1392,   716,   382,  4383,   986, 50256, 50256,
         50256, 50256]),
 tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 0, 0, 0, 0]))

## 5. Train

Forward: frozen transformer → last real token → `lm.lm_head` → cross-entropy. Optimizer sees only `lm.lm_head`.

In [ ]:
def forward(token_ids, mask):
    with torch.no_grad():
        hidden = lm.transformer(token_ids, attention_mask=mask).last_hidden_state  # (B, T, 768)

    # each row can have different length of token_id and masks : dynamic 
    last = hidden[torch.arange(token_ids.size(0)), mask.sum(1) - 1] 
    return lm.lm_head(last)  # (B, 2)


def run(loader, opt=None):
    lm.lm_head.train(opt is not None)

    total_loss, correct_answers, n = 0.0, 0, 0
    
    for token_ids, mask, y in loader:
        token_ids, mask, y = token_ids.to(device), mask.to(device), y.to(device)
        logits = forward(token_ids, mask)
        loss = nn.functional.cross_entropy(logits, y)
        if opt:
            opt.zero_grad()
            loss.backward()
            opt.step()
        total_loss += loss.item() * len(y)
        correct_answers += (logits.argmax(-1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct_answers / n


opt = torch.optim.AdamW(lm.lm_head.parameters(), lr=5e-4)
for epoch in range(1, 10):
    tr = run(train_loader, opt)
    va = run(val_loader)
    print(f"epoch {epoch}  train loss {tr[0]:.4f} acc {tr[1]:.3f}  val loss {va[0]:.4f} acc {va[1]:.3f}")

epoch 1  train loss 0.8808 acc 0.463  val loss 0.2371 acc 0.900
epoch 2  train loss 0.4845 acc 0.805  val loss 0.2239 acc 0.900
epoch 3  train loss 0.4398 acc 0.805  val loss 0.1988 acc 1.000
epoch 4  train loss 0.3851 acc 0.854  val loss 0.1848 acc 1.000
epoch 5  train loss 0.3654 acc 0.902  val loss 0.1873 acc 1.000
epoch 6  train loss 0.3422 acc 0.902  val loss 0.1707 acc 1.000
epoch 7  train loss 0.3090 acc 0.927  val loss 0.1614 acc 1.000
epoch 8  train loss 0.2826 acc 0.951  val loss 0.1515 acc 1.000
epoch 9  train loss 0.2576 acc 0.976  val loss 0.1417 acc 1.000


## 6. Try two messages

In [36]:
lm.lm_head.eval()

for text in [
    "I'll be home around 7, want me to pick up dinner?",
    "WINNER! Free £1000 prize. Call 09061701461 now."]:

    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LEN, padding="max_length")
    logits = forward(enc["input_ids"], enc["attention_mask"])[0]
    p = torch.softmax(logits, dim=-1) # p[0] for ham, p[1] for spam
    if int(p.argmax()) == 0:
        prediction = 'ham'
    else:
        prediction = 'spam'
    print(f"Prediction : {prediction}  Text {text}")

Prediction : ham  Text I'll be home around 7, want me to pick up dinner?
Prediction : spam  Text WINNER! Free £1000 prize. Call 09061701461 now.
